### 복습
- data폴더 안에 가전 폴더의 모든 데이터셋을 로드하여 하나의 데이터프레임으로 생성
- 감성(GeneralPolarity) 컬럼의 데이터를 -1, 0, 1에서 0, 1, 2로 변환
- RawText의 데이터에서 텍스트 정규화, 중복 데이터 제거, 글자 수가 1이하인 데이터 제거
- 감성이 결측치인 데이터를 따로 저장
- 원본의 데이터에서는 결측치를 제거
- GeneralPolarity 컬럼의 이름을 labels로 변경
- RawText, labels 컬럼만 유지
- train, test 데이터셋 분할(8:2)
- SBERT 모델을 생성 ('jhgan/ko-sroberta-multitask')
- Datset에서 SBERT 모델을 이용하여 임베딩 (생성자 함수에서 일괄 처리)
- Dataloader를 이용하여 batch_size = 128 배치 데이터 생성
- 다중 퍼셉트론 모델을 생성하여 감성분석 (Linear -> ReLU -> Dropout -> Linear)
- 반복 학습의 횟수 20
- 검증 데이터를 이용하여 f1_score를 확인
- 결측치 데이터에서 랜덤하게 5개 데이터를 추출하여 예측값을 확인
- 딥러닝 모델이 아닌 SVC 모델을 이용하여 f1_score 확인

In [1]:
import os 
import re 
import pandas as pd
import torch 
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import f1_score 
from sentence_transformers import SentenceTransformer
from sklearn.svm import SVC

c:\study\multicampus_practice\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def normalize(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\,]", ' ', str(text))
    text = re.sub(r"\s+", ' ', text).strip()
    return text

In [3]:
# os 라이브러리를 이용하여 파일의 목록을 로드 
# file_path 마지막에 / 사용하는 이유는? 파일이름과 경로를 연결해서 사용할때 편리하기 위함
file_path = "../data/가전/"
file_list = os.listdir(file_path)
file_list

['3-1.영상음향가전(76).json',
 '3-1.영상음향가전(77).json',
 '3-1.영상음향가전(78).json',
 '3-1.영상음향가전(79).json',
 '3-1.영상음향가전(80).json',
 '3-1.영상음향가전(81).json',
 '3-1.영상음향가전(82).json',
 '3-1.영상음향가전(83).json',
 '3-1.영상음향가전(84).json',
 '3-1.영상음향가전(85).json',
 '3-1.영상음향가전(86).json',
 '3-1.영상음향가전(87).json',
 '3-1.영상음향가전(88).json',
 '3-2.생활미용욕실가전(128).json',
 '3-2.생활미용욕실가전(129).json',
 '3-2.생활미용욕실가전(130).json',
 '3-2.생활미용욕실가전(131).json',
 '3-2.생활미용욕실가전(132).json',
 '3-2.생활미용욕실가전(133).json',
 '3-2.생활미용욕실가전(134).json',
 '3-2.생활미용욕실가전(135).json',
 '3-2.생활미용욕실가전(136).json',
 '3-2.생활미용욕실가전(137).json',
 '3-2.생활미용욕실가전(138).json',
 '3-2.생활미용욕실가전(139).json',
 '3-2.생활미용욕실가전(140).json',
 '3-3.주방가전(127).json',
 '3-3.주방가전(128).json',
 '3-3.주방가전(129).json',
 '3-3.주방가전(130).json',
 '3-3.주방가전(131).json',
 '3-3.주방가전(132).json',
 '3-3.주방가전(133).json',
 '3-3.주방가전(134).json',
 '3-3.주방가전(135).json',
 '3-3.주방가전(136).json',
 '3-3.주방가전(137).json',
 '3-3.주방가전(138).json',
 '3-3.주방가전(139).json',
 '3-4.계절가전(126).json',
 '3-4.계절가전(127)

In [4]:
df = pd.DataFrame()

for file in file_list:
    # file : 파일 이름 
    data = pd.read_json(file_path + file)
    df = pd.concat([df, data], axis=0)
df.info()

<class 'pandas.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   str    
 2   Source           4056 non-null   str    
 3   Domain           4056 non-null   str    
 4   MainCategory     4056 non-null   str    
 5   ProductName      4056 non-null   str    
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(1), str(5)
memory usage: 3.9+ MB


In [5]:
# 필요한 컬럼을 제외하고 나머지 컬럼을 삭제 
df = df[['RawText', 'GeneralPolarity']]

In [6]:
# 특정 컬럼의 이름을 변경 -> 객체 안에 변수를 직접 수정(df.columns), rename()함수를 이용 
df.rename(columns={
    'GeneralPolarity' : 'labels'
}, inplace=True)

In [7]:

# RawText 컬럼의 데이터를 텍스트 정규화 
df['RawText'] = df['RawText'].map(normalize)

In [8]:
# 길이가 1 이하인 데이터는 제외 
flag = df['RawText'].str.len() > 1
df = df.loc[flag, ]

In [9]:
# RawText의 중복 데이터를 제거 
df.drop_duplicates('RawText', inplace=True)

In [10]:
# labels의 결측치가 존재 -> 결측치만 따로 저장 
df_na = df.loc[df['labels'].isna(), ]
df2 = df.loc[~(df['labels'].isna()), ]

In [11]:
df2.info()

<class 'pandas.DataFrame'>
Index: 3678 entries, 0 to 99
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  3678 non-null   str    
 1   labels   3678 non-null   float64
dtypes: float64(1), str(1)
memory usage: 3.0 MB


In [12]:
df_na.info()

<class 'pandas.DataFrame'>
Index: 378 entries, 13 to 88
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  378 non-null    str    
 1   labels   0 non-null      float64
dtypes: float64(1), str(1)
memory usage: 322.8 KB


In [13]:
df2['labels'].value_counts()

labels
 1.0    2220
 0.0     944
-1.0     514
Name: count, dtype: int64

In [14]:
# labels 컬럼의 데이터 타입을 int 변경 -> 
# 딥러닝 모델에서 분류 모델을 생성할 때 3개의 컬럼의 수치가 등록 -> 가장 높은 값을 가진 위치가 정답 
# 위치 값은 int -> 위치는 0부터 시작 
df2['labels'] = df2['labels'].astype(int)

In [15]:
df2['labels'] = df2['labels'].map(
    {
        -1 : 0, 
        0 : 1, 
        1 : 2
    }
)

In [16]:
# df2['labels'] + 1

In [17]:
df2['labels'].value_counts()

labels
2    2220
1     944
0     514
Name: count, dtype: int64

In [18]:
# train, test 데이터셋 분할 
train_df, test_df = train_test_split(
    df2, test_size = 0.2, random_state = 42, stratify = df2['labels']
)

In [19]:
train_df['labels'].value_counts()

labels
2    1776
1     755
0     411
Name: count, dtype: int64

In [20]:
model_name = 'jhgan/ko-sroberta-multitask'
sbert = SentenceTransformer(model_name)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4015.21it/s]


In [21]:
# 최대 토큰 길이 제한 
sbert.max_seq_length = 128

In [22]:
class SBERTDataset(Dataset):
    def __init__(self, text, labels):
        # 생성자 함수에서 임베딩 처리를 완료 
        # class 생성시 데이터의 모든 임베딩 처리를 완료 -> class가 생성이 되는 과정에서는 시간이 걸리지만 
        # 딥러닝 반복 학습에서 시간이 단축 ( 메모리의 사용량은 증가 )
        # 대용량의 데이터를 이용해서 Dataset을 생성하는 경우에는 OOM(Out Of Memory)문제가 발생할 수 있다.
        with torch.inference_mode():
            # encode() : 토큰화 -> 인코딩 -> 벡터화(스케일링)
            self.emb = sbert.encode(
                text, convert_to_tensor=True, normalize_embeddings=True
            )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.emb[idx], self.labels[idx]

In [ ]:
train_ds = SBERTDataset(train_df['RawText'].tolist(), train_df['labels'].tolist())
test_ds = SBERTDataset(test_df['RawText'].tolist(), test_df['labels'].tolist())

In [ ]:
train_dl = DataLoader(train_ds, batch_size = 128, shuffle = True)
test_dl = DataLoader(test_ds, batch_size=128, shuffle = True)

In [ ]:
# 다중 퍼셉트론 구조의 분류 모델을 선언 
class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden_dim = 256, num_classes = 2, dropout = 0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), 
            nn.ReLU(), 
            nn.Dropout(dropout), 
            nn.Linear(hidden_dim, num_classes)
        )
    # 순전파 함수 
    def forward(self, X):
        result = self.net(X)
        return result

In [ ]:
# sbert 모델에서 출력 차원의 개수를 확인 
in_dim = sbert.get_sentence_embedding_dimension()
in_dim

In [ ]:
# 모델 생성
model = MLPModel(in_dim, num_classes=3)
# 손실 함수 
# 데이터의 불균형 존재 -> 샘플링 기법, ML 모델에서는 class_weight 매개변수를 이용해서 밸런스를 맞춰주는 방법
# DL 모델에서는 손실함수에서 클래스별 패널티를 지정 (weight 매개변수) 
# [0, 1, 2] -> 데이터가 적을수록 높은 변화 
# 5.0 ~ 10.0 : 데이터의 비율이 굉장히 적은 경우
# 2.0 ~ 4.0 : 데이터의 비율이 적은 경우
# 1.0 : 가장 많은 데이터
criterion = nn.CrossEntropyLoss(weight=torch.tensor([5.0, 3.0, 1.0]))
# 옵티마이저 
optimizer = optim.AdamW(model.parameters(), lr = 2e-04)

In [ ]:
# 학습 모드 전환
model.train()

for epoch in range(20):
    total = 0.0
    for X, y in train_dl:
        # 기울기 초기화
        optimizer.zero_grad()
        # 예측값 생성
        logits = model(X)
        # 손실 함수 계산
        loss = criterion(logits, y)
        # 가중치 연산
        loss.backward()
        # 가중치 변화
        optimizer.step()

        total += loss.item() * X.size(0)
    if (epoch+1) % 5 == 0:
        print(f"{epoch + 1} : loss : {round(total/len(train_df), 4)}" )

In [ ]:
# 예측 값 생성 
model.eval()

y_true , y_pred=  [], []

with torch.inference_mode():
    for X, y in test_dl:
        logits = model(X)
        pred = logits.argmax(dim = -1).tolist()
        y_true.extend(y.tolist())
        y_pred += pred

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
print(accuracy_score(y_true, y_pred))
print(f1_score(y_true, y_pred, average='macro'))

In [ ]:
# 결측치 데이터중 랜덤하게 5개를 추출 
samples = df_na['RawText'].sample(5).tolist()
samples

In [ ]:
@torch.no_grad()
def predict_review(text, batch_size = 128):
    # samples 데이터에서 임베딩 처리 
    sbert.eval()
    model.eval()

    result = []
    # 임의로 배치 생성
    for idx in range(0, len(text), batch_size):
        # 배치 구간을 생성
        batch_text = text[idx : idx + batch_size]

        # 배치 구간을 encode() 대입 
        embs = sbert.encode(
            batch_text, convert_to_tensor=True, normalize_embeddings=True
        )
        # 예측 값 생성
        logits = model(embs)
        # 예측 값을 이용하여 확률로 변환
        probs = logits.softmax(dim = -1)
        # 예측 위치 계산
        preds = probs.argmax(dim = -1).tolist()
        # 예측 값을 변환하는 dict
        label_to_id = {
            0 : '부정', 
            1 : '중립', 
            2 : '긍정'
        }

        for idx2, pred in enumerate(preds):
            # idx : 전체 문장에서  배치의 시작 위치
            # idx2 : 배치 안에서의 초기화된 위치 값
            # idx + idx2 -> 전체 문장에서 해당 문장의 위치
            review = text[idx + idx2]
            # 예측 확률
            prob = float(probs[idx2, pred])
            # 예측 값에 따른 id로 변환 
            label = label_to_id[pred]

            result.append(
                {
                    'text' : review, 
                    'prob' : prob, 
                    'label' : label
                }
            )
    return result

In [ ]:
pd.DataFrame(predict_review(samples))

In [ ]:

svc = SVC(random_state=42)

In [ ]:
train_ds[0:len(train_ds)]

In [ ]:

X_train, y_train = train_ds[0:len(train_ds)]
X_test, y_test = test_ds[0:len(test_ds)]

In [ ]:
import numpy as np

In [ ]:

svc.fit(np.array(X_train), np.array(y_train))

In [ ]:
pred = svc.predict(np.array(X_test))

In [ ]:
f1_score(pred, np.array(y_test), average='macro')